In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit


In [3]:
import os
os.getcwd()


'c:\\Users\\USER\\OneDrive - Efrei\\Documents\\Cours MSE\\file rouge\\fil_rouge-etude_chomage_france\\src\\features'

In [6]:
# Chargement
df = pd.read_csv("../../data/processed/02_dataset_clean.csv")

# Date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

df.shape
TARGET = "taux_chomage_total_insee"

X = df.drop(columns=[TARGET, "date"])
y = df[TARGET]

X.shape, y.shape

X_imp = X.fillna(X.median())
y_imp = y.fillna(y.median())
mi = mutual_info_regression(X_imp, y_imp, random_state=42)




ValueError: Input X contains infinity or a value too large for dtype('float64').

In [7]:
from sklearn.feature_selection import mutual_info_regression

mi = mutual_info_regression(X_imp, y_imp, random_state=42)
mi_rank = (
    pd.Series(mi, index=X.columns)
    .sort_values(ascending=False)
)

mi_rank


population_active                       3.064031
annee                                   3.039246
pib                                     2.517611
ict                                     2.286598
demandeur_femme_abcd_plus50             2.223257
demandeur_total_abcd_plus50             1.979398
demandeur_homme_abcd_plus50             1.815927
taux_euribor_3m                         1.772830
demandeur_total_abcd_total              1.713239
demandeur_homme_abcd_total              1.677301
demandeur_femme_abcd_total              1.617897
taux_chomage_femme_insee                1.594326
taux_chomage_total_insee                1.592527
taux_chomage_ocde                       1.571845
demandeur_homme_abcd_2549               1.564296
taux_chomage_25_49_insee                1.549909
demandeur_total_abcd_2549               1.545044
MRO                                     1.503367
demandeur_femme_abcd_2549               1.464903
demandeur_total_abcd_moins25            1.422197
demandeur_femme_abcd

In [8]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

lasso_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(
        cv=tscv,
        random_state=42,
        max_iter=20000
    ))
])


In [9]:
lasso_pipeline.fit(X, y)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('lasso',
                 LassoCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                         max_iter=20000, random_state=42))])

In [10]:
coef = lasso_pipeline.named_steps["lasso"].coef_

lasso_rank = (
    pd.Series(coef, index=X.columns)
    .sort_values(key=abs, ascending=False)
)

lasso_rank


taux_chomage_total_insee                0.592596
taux_chomage_25_49_insee                0.286960
taux_chomage_15_24_insee                0.067593
taux_chomage_ocde                       0.061866
pib                                    -0.040580
MRO                                    -0.027803
annee                                  -0.000000
isj                                    -0.000000
population_active                      -0.000000
taux_euribor_3m                         0.000000
ipc                                     0.000000
indicateur_climat_emploi                0.000000
ipc_energie_only                        0.000000
indicateur_climat_affaires             -0.000000
demandeur_femme_abcd_plus50            -0.000000
indicateur_retournement_conjoncturel    0.000000
ict                                    -0.000000
nb_defaillances_entreprise              0.000000
nb_offres_france_travail               -0.000000
demandeur_femme_abcd_moins25            0.000000
demandeur_femme_abcd